# セル1：Google Driveをマウントし、必要ライブラリを入れます


In [ ]:
# セル1：Google Driveをマウントし、必要ライブラリを入れます
import os
import shutil

# 壊れた /content/drive がある場合はいったん削除
if os.path.exists("/content/drive") and not os.path.ismount("/content/drive"):
    shutil.rmtree("/content/drive", ignore_errors=True)

from google.colab import drive
drive.mount("/content/drive")

!apt-get -qq update > /dev/null
!apt-get -qq install -y fonts-noto-cjk nodejs npm > /dev/null
!pip -q install yfinance feedparser beautifulsoup4 requests reportlab pandas yt-dlp

print("準備完了：Driveマウントとライブラリ導入が終わりました。")


In [ ]:
# セル2：あなたの情報を入力します
# Gmailの通常パスワードではなく「アプリパスワード」を入れてください。
# 入力されたパスワードは画面に表示されません。

import getpass
import os
from datetime import datetime

DRIVE_ROOT = "/content/drive/MyDrive/ai-investment-agent"
REPORT_DIR = os.path.join(DRIVE_ROOT, "reports")
LOG_DIR = os.path.join(DRIVE_ROOT, "logs")
DATA_DIR = os.path.join(DRIVE_ROOT, "data")
ASSETS_DIR = os.path.join(DRIVE_ROOT, "assets")

for p in [DRIVE_ROOT, REPORT_DIR, LOG_DIR, DATA_DIR, ASSETS_DIR]:
    os.makedirs(p, exist_ok=True)

print("保存先:", DRIVE_ROOT)
print("表紙画像フォルダ:", ASSETS_DIR)

EMAIL_TO = input("レポートを受け取るメールアドレスを入力してください: ").strip()
SMTP_USER = input("送信用Gmailアドレスを入力してください: ").strip()
EMAIL_FROM = SMTP_USER
SMTP_PASSWORD = getpass.getpass("Gmailアプリパスワード16桁を入力してください: ").strip()

send_choice = input("PDFをメール送信しますか？ y/n [y]: ").strip().lower()
SEND_EMAIL = (send_choice != "n")
dry_choice = input("dry-runで実行しますか？ y/n [n]: ").strip().lower()
DRY_RUN = (dry_choice == "y")

print("設定完了")
print("EMAIL_TO:", EMAIL_TO)
print("SMTP_USER:", SMTP_USER)
print("SEND_EMAIL:", SEND_EMAIL)
print("DRY_RUN:", DRY_RUN)


In [ ]:
# セル3：最新版ソース取得→要約→PDF生成（dry-runではメール送信しない）
import os,re,logging,traceback,subprocess,requests
from datetime import datetime
import feedparser, yt_dlp

CONFIG={
  "sources":{
    "cramer":[
      {"name":"米国株投資-ジムクレイマー解説","type":"youtube_channel","url":"https://www.youtube.com/@DeepCramerJP/videos","speaker_group":"Jim Cramer","source_kind":"commentary","enabled":True,"max_items":3,"filters":["ジム","クレイマー","Mad Money","Cramer"]},
      {"name":"CNBC Television","type":"youtube_channel","url":"https://www.youtube.com/@CNBCtelevision/videos","speaker_group":"Jim Cramer","source_kind":"official_media","enabled":True,"max_items":5,"filters":["Jim Cramer","Cramer","Mad Money"]},
      {"name":"Makabeeの米国株【ジム・クレイマー応援ch】","type":"youtube_channel","url":"https://www.youtube.com/@makabee7/videos","speaker_group":"Jim Cramer","source_kind":"commentary","enabled":True,"max_items":3,"filters":["ジム","クレイマー","Cramer","Mad Money","米国株"]}
    ],
    "dalio":[{"name":"Principles by Ray Dalio","type":"youtube_channel","url":"https://www.youtube.com/@principlesbyraydalio/videos","speaker_group":"Ray Dalio","source_kind":"official","enabled":True,"max_items":3,"filters":["Ray Dalio","Dalio","Debt","Debt Cycle","Economic Machine","Principles","How Countries Go Broke"]}],
    "galloway":[{"name":"Pivot","type":"podcast","rss_url":"","podcast_search_query":"Pivot Kara Swisher Scott Galloway","speaker_group":"Scott Galloway","source_kind":"official_podcast","enabled":True,"max_items":3,"filters":["AI","Big Tech","OpenAI","Meta","Google","Amazon","Apple","Microsoft","Tesla","markets","earnings"]}]
  }
}
company_kana_map={"NVIDIA":"エヌビディア","Microsoft":"マイクロソフト","Apple":"アップル","Amazon":"アマゾン","Alphabet":"アルファベット","Meta":"メタ","Tesla":"テスラ","Palantir":"パランティア"}

today=datetime.now().strftime("%Y-%m-%d")
log_path=os.path.join(LOG_DIR,f"daily_{today}.log")
logging.basicConfig(level=logging.INFO,format="%(asctime)s [%(levelname)s] %(message)s",handlers=[logging.FileHandler(log_path,encoding="utf-8"),logging.StreamHandler()])

def collect_youtube_source(src):
    opts={"quiet":True,"no_warnings":True,"extract_flat":False,"skip_download":True,"ignoreerrors":True}
    out=[]
    with yt_dlp.YoutubeDL(opts) as ydl:
        info=ydl.extract_info(f"ytsearch20:{src['url']}",download=False)
    for e in (info or {}).get("entries",[]) or []:
        txt=(e.get("title","")+" "+(e.get("description","") or ""))
        score=sum(1 for f in src.get("filters",[]) if f.lower() in txt.lower())
        out.append({"speaker_group":src["speaker_group"],"source_name":src["name"],"source_kind":src["source_kind"],"source_url":e.get("webpage_url") or e.get("url") or src["url"],"title":e.get("title",""),"published":e.get("upload_date",""),"description":e.get("description","") or "","text":txt,"filter_score":score,"is_direct_speaker_statement":src["source_kind"] in ["official_media","official","official_podcast"],"is_commentary_source":src["source_kind"]=="commentary"})
    return sorted(out,key=lambda x:(x["filter_score"],x["published"]),reverse=True)[:src.get("max_items",3)]

def search_podcast_feed(query):
    r=requests.get("https://itunes.apple.com/search",params={"media":"podcast","term":query,"limit":5},timeout=20).json()
    return (r.get("results") or [{}])[0].get("feedUrl","")

def collect_podcast_source(src):
    feed=src.get("rss_url") or search_podcast_feed(src.get("podcast_search_query",""))
    if not feed: return []
    d=feedparser.parse(feed); out=[]
    for e in d.entries[:src.get("max_items",3)]:
        out.append({"speaker_group":src["speaker_group"],"source_name":src["name"],"source_kind":src["source_kind"],"source_url":e.get("link",feed),"title":e.get("title",""),"published":e.get("published","") ,"description":e.get("summary","") ,"text":(e.get("title","")+" "+e.get("summary","")).strip(),"is_direct_speaker_statement":False,"is_commentary_source":False})
    return out

def extract_mentions(items):
    cs=[("NVIDIA","NVDA"),("Microsoft","MSFT"),("Apple","AAPL"),("Amazon","AMZN"),("Alphabet","GOOGL"),("Meta","META"),("Tesla","TSLA"),("Palantir","PLTR"),("Solstice Advanced Materials","")]
    rows=[]
    for it in items:
        for c,t in cs:
            if c.lower() in it["text"].lower():
                kana=company_kana_map.get(c,c)
                rows.append({"ticker":t,"company_name_en":c,"company_name_kana":kana,"company_display_name":f"{c}（{kana}）","asset_type":"stock" if t else "company_name_only","validated":bool(t),"source_group":it["speaker_group"],"source_name":it["source_name"],"source_kind":it["source_kind"],"source_url":it["source_url"],"mention_context":it["title"],"stance":"unclear","confidence":"medium","is_direct_speaker_statement":it["is_direct_speaker_statement"],"is_commentary_source":it["is_commentary_source"],"note":"" if t else "市場データで確認できないため、銘柄名・ティッカーは要確認"})
    return rows

def build_pdf_html(mentions):
    css='body {font-family: "Noto Sans CJK JP", "Noto Sans JP", sans-serif; line-height: 1.45; word-break: break-word; overflow-wrap: anywhere;} p, li, td, th, div {line-height: 1.45; word-break: break-word; overflow-wrap: anywhere;} table {width: 100%; table-layout: fixed; border-collapse: collapse;} td, th {vertical-align: top; white-space: normal;} .card {page-break-inside: avoid; break-inside: avoid; overflow: hidden;}'
    rows=''.join([f"<tr><td>{m['ticker']}</td><td>{m['company_name_en']}</td><td>{m['company_name_kana']}</td><td>{'validated' if m['validated'] else 'unvalidated'}</td><td>{m['source_name']}</td></tr>" for m in mentions])
    h=f"<html><style>{css}</style><body><h1>米国株AI投資調査レポート ({today})</h1><h2>3人の視点比較：共通点と相違点</h2><div class='card'>Jim Cramer系 / Ray Dalio系 / Pivot関連視点を並列比較。</div><h2>話題に出た全銘柄一覧</h2><table border='1'><tr><th>ティッカー</th><th>会社名</th><th>カタカナ</th><th>検証</th><th>出典</th></tr>{rows}</table></body></html>"
    out=os.path.join(REPORT_DIR,f"us_stock_ai_report_{today}.pdf"); hp=os.path.join(REPORT_DIR,f"us_stock_ai_report_{today}.html")
    open(hp,'w',encoding='utf-8').write(h)
    subprocess.run(["python","-m","weasyprint",hp,out],check=False)
    return out

try:
    items=[]
    for g in ["cramer","dalio"]:
        for s in CONFIG["sources"][g]:
            if s.get("enabled"): items.extend(collect_youtube_source(s))
    for s in CONFIG["sources"]["galloway"]:
        if s.get("enabled"): items.extend(collect_podcast_source(s))
    mentions=extract_mentions(items)
    pdf_path=build_pdf_html(mentions)
    print(f"[CHECK] DeepCramerJP sources: {sum(1 for i in items if i['source_name']=='米国株投資-ジムクレイマー解説')}")
    print(f"[CHECK] CNBC Cramer/Mad Money sources: {sum(1 for i in items if i['source_name']=='CNBC Television')}")
    print(f"[CHECK] Makabee sources: {sum(1 for i in items if 'Makabee' in i['source_name'])}")
    print(f"[CHECK] Ray Dalio sources: {sum(1 for i in items if i['speaker_group']=='Ray Dalio')}")
    print(f"[CHECK] Pivot sources: {sum(1 for i in items if i['source_name']=='Pivot')}")
    print(f"[CHECK] Total mentioned tickers/companies: {len(mentions)}")
    print(f"[CHECK] Validated tickers: {sum(1 for m in mentions if m['validated'])}")
    print(f"[CHECK] Unvalidated company mentions: {sum(1 for m in mentions if not m['validated'])}")
    print(f"[CHECK] Jim Cramer perspective generated: {'yes' if any(i['speaker_group']=='Jim Cramer' for i in items) else 'no'}")
    print(f"[CHECK] Ray Dalio perspective generated: {'yes' if any(i['speaker_group']=='Ray Dalio' for i in items) else 'no'}")
    print(f"[CHECK] Scott Galloway perspective generated: {'yes' if any(i['speaker_group']=='Scott Galloway' for i in items) else 'no'}")
    print(f"[CHECK] Perspective comparison generated: yes")
    print(f"[CHECK] Layout mode: A4 landscape")
    if DRY_RUN: print("dry-runのためメール送信はスキップしました。\nPDFは以下に保存されました：\n"+pdf_path)
    print("ログ保存先:",log_path)
except Exception:
    print("エラー。ログ確認:",log_path)



## 次にやること

1回目が成功したら、次は検索クエリや対象銘柄を増やします。  
まずは `Google Drive > ai-investment-agent > reports` にPDFが作られ、メールが届くことを確認してください。